In [1]:
import os
import re

import numpy as np
from numpy.random import randint, normal, uniform
import pylab as pl
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from astropy import units as u

import healpy as hp
import camb

from ksw import Cosmology, Data, radial_functional as rf

%matplotlib inline

These are the functions that compute $a_{lm}^{NG,loc'}$ taken from [Hanson](https://arxiv.org/abs/0905.4732)

In [2]:
def interpolate_ells(func, ells_sparse, ls):
    # ells_sparse are the ells used in the transfer function
    # ls are the ells used in the healpy alms
    # func is a function of ells_sparse
    # returns (nr, nls, npol)
    gen = interp1d(ells_sparse, func, kind='cubic', axis=1, bounds_error=False, fill_value="extrapolate")
    return gen(ls)

def get_delta_phi():
    # primordial normalization
    return 2 * np.pi ** 2 * 2.13e-09 * (3 / 5)**2

def get_alpha_l(tr_ell_k, ks, radii, ells):
    # radial func does f_ell^X(r) = (2/pi) int k^2 dk f(k) transfer^X_ell(k) j_ell(k r)
    # alpha_l = (2/pi) int k^2 dk transfer^X_ell(k) j_ell(k r)
    # returns shape (nr, nell, npol)
    
    f_k = np.ones((len(ks), 1))
    return rf.radial_func(f_k, tr_ell_k, ks, radii, ells).squeeze()

def get_beta_l(tr_ell_k, ks, radii, ells):
    # radial func does f_ell^X(r) = (2/pi) int k^2 dk f(k) transfer^X_ell(k) j_ell(k r)
    # beta_l = (2/pi) int k^-1 dk delta_phi(k) transfer^X_ell(k) j_ell(k r)
    # returns shape (nr, nell, npol)

    f_k = np.swapaxes([ks**-3 * get_delta_phi()], 0, 1)
    return rf.radial_func(f_k, tr_ell_k, ks, radii, ells).squeeze()

def a_lm_ng(alms, data, nside, pol, r_max=15000, r_res=1000):
    lmax_alm = hp.Alm.getlmax(alms.shape[-1])

    # These values are sparse in ell
    c_ells = data.cosmology.c_ell['unlensed_scalar']['ells']
    tr_ell_k = data.cosmology.transfer['tr_ell_k']
    ells = data.cosmology.transfer['ells']
    ks = data.cosmology.transfer['k']

    radii = np.linspace(1, r_max, r_res)
    alpha_ell = get_alpha_l(tr_ell_k, ks, radii, ells)
    beta_ell = get_beta_l(tr_ell_k, ks, radii, ells)

    # interpolate to the dense ells used in the alms
    ls, _ = hp.Alm.getlm(lmax_alm)
    alpha_l = interpolate_ells(alpha_ell, ells, ls)
    bl_div_cl = interpolate_ells(beta_ell / c_ells[None, ells, None], ells, ls)

    print('Starting calculation...', end=' ')
    dr = radii[1] - radii[0]
    alm_ng = np.empty((radii.shape[0], alms.shape[0], alms.shape[1]), dtype=np.complex128)
    for r in range(radii.shape[0]):
        for p in range(alms.shape[0]):
            Balm = hp.almxfl(alms[p], bl_div_cl[r, :, p]) # beta_l / c_ell * a_lm
            B = hp.alm2map(Balm, nside=nside, lmax=lmax_alm, mmax=None, pol=pol, pixwin=False, fwhm=0, sigma=None)

            inner = hp.map2alm(B**2, lmax=lmax_alm, mmax=None, pol=pol)
            alm_ng[r, p] = dr * radii[r]**2 * alpha_l[r, :, p] * inner

        if r % 100 == 0:
            print(r, end=' ')
    print('Done!')
    return np.sum(alm_ng, axis=0)

In [3]:
def cutSqPatches(fullsky_map, img_size, side_deg, num_patches):
    Tmap_datat = np.zeros((num_patches//2, int(img_size), int(img_size)))
    Tmap_datab = np.zeros((num_patches//2, int(img_size), int(img_size)))

    pl.ioff()
    for counter in range(num_patches//2):
        Tmap_datat[counter] = np.ma.getdata(hp.cartview(fullsky_map, fig=0, xsize=img_size, ysize=img_size, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[0, side_deg],
                                                        title="CartView", unit="mK", format="%.2g", return_projected_map=True))
        Tmap_datab[counter] = np.ma.getdata(hp.cartview(fullsky_map, fig=1, xsize=img_size, ysize=img_size, rot=[0, 0], lonra=[side_deg*counter, side_deg*(counter+1)], latra=[-side_deg, 0],
                                                        title="CartView", unit="mK", format="%.2g", return_projected_map=True))
    pl.close('all')
    pl.ion()
    
    return np.concatenate((Tmap_datat, Tmap_datab))

In [4]:
def run_simulations(num_sim, 
                    cosmo_params, 
                    fnl_range, 
                    lmax, 
                    nside,
                    side_deg, 
                    num_patches, 
                    polarizations,
                    r_max,
                    r_res,
                    beam_width,
                    noise_loc,
                    noise_scale):
    
    print(f"Running {num_sim} simulations with fnl in range {fnl_range}, and polarizations {polarizations}...")
    print("Init Cosmology...")
    camb_params_obj = camb.set_params(**cosmo_params)
    cosmo = Cosmology(camb_params_obj, verbose=True)
    cosmo.compute_transfer(cosmo_params['max_l'], verbose=True)
    cosmo.compute_c_ell()
    print("Done!")

    npol = 1 if isinstance(polarizations, str) else len(polarizations)
    pol = npol > 1
    nell = lmax + 1

    n_ell = normal(noise_loc, noise_scale, (3, nell) if pol else (nell))                          # noise
    b_ell = hp.gauss_beam(beam_width.to_value(u.radian), lmax, pol)
    data = Data(lmax, n_ell, b_ell, polarizations, cosmo) 
    
    print("Computing alm...", end=' ')
    alm = data.compute_alm_sim(lens_power=False)
    print("Done!")
    
    print("Computing a_lm_ng...", end=' ')
    alm_ng = a_lm_ng(alm, data, nside, pol, r_max, r_res)

    print("Running simulations...")
    fnls = uniform(fnl_range[0], fnl_range[1], num_sim).astype(np.float32)
    patches = np.empty((npol, num_sim, num_patches, nside, nside))
    maps = np.empty((npol, num_sim, 12*nside**2))

    for i in range(num_sim):
        alm_prime = alm + fnls[i] * alm_ng
        for p in range(npol):
            maps[p, i]  = hp.alm2map(alm_prime[p], nside, lmax=lmax, mmax=None, pol=pol, pixwin=False, fwhm=0, sigma=None)
            patches[p, i] = cutSqPatches(maps[p, i], nside, side_deg, num_patches)
        
        if i % 100 == 0:
            plt.figure()
            hp.mollview(maps[0, i], title=f"sim {i} [fnl={fnls[i]}]")
            plt.show()
            
        if i % 100 == 0:
            print(i, end=' ')
            
    print("Done!")
    return (fnls, patches, maps)

See: https://camb.readthedocs.io/en/latest/model.html

Some values are forced during init of ksw.Cosmology. 
For an example on how to modify these values after init: https://github.com/AdriJD/ksw/blob/fba3250cfe4c5145b5db12fa54bbc54bbcf5a56c/tests/python/test_cosmology.py#L65

In [5]:
cosmo_params = {
    'H0': 67.5,
    'r': 0,
    'As': 2.13e-09,
    'ns': 0.9624,
    'pivot_scalar': 0.05,
    'ombh2': 0.02233,
    'omch2': 0.1198,
    'mnu': 0.06,
    'tau': 0.0561,
    'TCMB': 2.7255,
    'max_l': 3000, # should be higher then lmax below, will get c_l_max > lmax error otherwise, for some reason
}

r_res, nside are probably the only ones to meaningfully effect preformance. Might have memory issues with too high values, we can batch if that becomes a problem.

lmax >= 300 is enforced by the ksw code due to errors with CAMB. 
lmax needs to be somewhat smaller then max_l, if you get errors about c_ell change these 

nside should be of type 2\*\*n

num_patches should be even.

patch_side_deg \* num_patches \<\= 180, patch_side_deg \<\= 45; or you will overlap patches 

r_max is given in Mpc, we preform r_res slices over this volume to do the integral. Values have been picked by guestimation, with r_res being set low to help test. Will want to raise this for better calculations.

KSW only supports values of 'T', 'E', ['T', 'E'].

In [6]:
# You will get a warning if lmax > 4*nside
lmax = 1500 # used for computing, max_l is for internal

fnl_range=(-1000, 1000)

nsims = 100 
npatches = 10
nside=256
patch_side_deg = 10

r_max = 14000               # Mpc, max radius for the patch
r_res = 100000              # number of slices in the radius

# KSW only supports 'T' and 'E'
polarizations = ['T']
# polarizations = ['T', 'E']

# FWHM of gaussian beam
beam_width = 1 * u.arcmin # type: ignore

# noise settings, Noise covariance matrix (without beam) in uK^2.
noise_loc = 0.
noise_scale = 1.

In [7]:
fnls, patches, maps = run_simulations(nsims, 
                                cosmo_params, 
                                fnl_range, 
                                lmax, 
                                nside,
                                patch_side_deg, 
                                npatches,
                                polarizations,
                                r_max,
                                r_res,
                                beam_width,
                                noise_loc,
                                noise_scale)

Running 100 simulations with fnl in range (-1000, 1000), and polarizations ['T']...
Init Cosmology...
Updated CAMB param: DoLateRadTruncation from True to False.
Updated CAMB param: AccuracyBoost from 1.0 to 2.
Updated CAMB param: BessIntBoost from 1.0 to 30.
Updated CAMB param: KmaxBoost from 1.0 to 3.
Updated CAMB param: IntTolBoost from 1.0 to 4.
Updated CAMB param: TimeStepBoost from 1.0 to 4.
Updated CAMB param: SourcekAccuracyBoost from 1.0 to 5.
Updated CAMB param: BesselBoost from 1.0 to 5.
Updated CAMB param: IntkAccuracyBoost from 1.0 to 5.
Updated CAMB param: lSampleBoost from 1.0 to 2.
Updated CAMB param: lAccuracyBoost from 1.0 to 2.
Updated CAMB param: AccurateBB from False to True.


In [ ]:
random_indices = [(0, randint(nsims), randint(npatches)) for _ in range(4)]
random_indices

[(0, 83, 8), (0, 77, 8), (0, 5, 7), (0, 4, 7)]

Lets plot a few patches

In [ ]:
for pol, s, p in random_indices:
    plt.imshow(patches[pol, s, p])
    plt.title(f"sim {s*npatches + p} [fnl={fnls[s]}]")
    plt.show()

Here we plot the C_ls

In [ ]:
for pol, s, p in random_indices:
    cl = hp.anafast(maps[pol, s, p], lmax=lmax)
    ell = np.arange(len(cl))
    plt.plot(ell, ell * (ell + 1) * cl)
    plt.title(f"sim {s*npatches + p} [fnl={fnls[s]}]")
    plt.show()

We will save the data below. Name is set by settings so it can be loaded easy by the model trainer.

Data output is
```
{
    'fnls': array((nsims)),
    'patches': array((npol, nsims, npatchs, nside, nside)),
    'maps': array((npol, nsims, 12*nside**2))
}
```
maps are the full healpy maps. 

The order of everything is set by nsims, with fnls[i] being used to generate the corresponding maps and patches.
Make sure you preserve this ordering.

We may want to optimize this with TFDatasets if we find GPU is idle a lot, which would indiciate data bound due to transfer.

In [ ]:
base_name = f'{nside}-{nsims}x{npatches}_fnl{fnl_range[0]}-{fnl_range[1]}-{polarizations}'
data_dir = f'data/ksw/'
data_filename = f'{base_name}'

data_dir, data_filename

In [ ]:
def _save(dir, filename, data):
    if not os.path.exists(dir):
        os.makedirs(dir)
    print(f"Saving to {dir}/{filename}...", end=" ")
    with open(f"{dir}/{filename}", "xb") as f:
        np.save(f, data)
    print("Done!")

# Create data dir if it doesn't exist
# If it exists we clean up the data files to prevent issues
if not os.path.exists(data_dir): 
    os.makedirs(data_dir)
    print(f'Created directory {data_dir}')
else:
    print(f'Reusing directory {data_dir}')
    # pattern = re.compile(f"{base_name}_\d+-\d+\.npy")
    pattern = re.compile(f"{base_name}.npy")
    for file in os.listdir(data_dir):
        if pattern.match(file):
            file_path = os.path.join(data_dir, file)
            os.remove(file_path)
            print(f"Deleted existing data file: {file_path}")

_save(data_dir, f'{data_filename}.npy', {'fnls':fnls, 'patches':patches, 'maps': maps})
print("Done!")